# 00 — Setup Verification
**Run this notebook once** after your very first session to confirm all tools work.

---
**This notebook is fully self-contained** — just run all cells top to bottom.
It will mount Drive, install everything, and verify the full pipeline works.

## 1. Mount Google Drive

In [21]:
from google.colab import drive
import time

drive.mount('/content/drive')
time.sleep(5)  # wait for Drive to fully sync

import os
PROJECT_ROOT = '/content/drive/MyDrive/thesis'
%cd {PROJECT_ROOT}
print(f'✅ Drive mounted')
print(f'✅ Working directory: {os.getcwd()}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/thesis
✅ Drive mounted
✅ Working directory: /content/drive/MyDrive/thesis


## 2. Install Dependencies
> Takes ~5 minutes on first run. Do not restart runtime when it finishes.

In [ ]:
# Core ML / HuggingFace
!pip install -q sentence-transformers accelerate safetensors bitsandbytes

# German NLP
!pip install -q stanza

# Topic modeling
!pip install -q bertopic hdbscan umap-learn
!pip install -q gensim

# Translation
!pip install -q easynmt sacremoses sentencepiece

# Evaluation metrics
!pip install -q bert-score sacrebleu
!pip install -q entmax jsonargparse==3.13.1
# Statistical validation
!pip install -q pingouin krippendorff

# Text utilities
!pip install -q ftfy langdetect textstat

# Project utilities
!pip install -q python-dotenv pyyaml rich jsonlines

# Fix numpy — some packages above downgrade it, spaCy needs 2.x
!pip install -q 'numpy>=2.0'

# spaCy German model
!python -m spacy download de_core_news_lg

import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')  # also commonly needed by EasyNMT

print('\n✅ All dependencies installed — do NOT restart runtime')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 567.8/567.8 MB 161.9 MB/s eta 0:00:01

## 3. Load Config & Set Random Seed

In [ ]:
import yaml, random, numpy as np, torch

with open(f'{PROJECT_ROOT}/config.yaml') as f:
    config = yaml.safe_load(f)

SEED = config['project']['random_seed']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'✅ Config loaded')
print(f'   Seed              : {SEED}')
print(f'   Translation model : {config["translation"]["primary_model"]}')
print(f'   LLM model         : {config["llm"]["primary_model"]}')
print(f'   Validation n      : {config["sampling"]["total_n"]}')


## 4. GPU Check

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')


## 5. Import Verification

In [ ]:
from rich.console import Console
from rich.table import Table

console = Console()
results = []

def try_import(module_name, display_name):
    try:
        mod = __import__(module_name)
        version = getattr(mod, '__version__', 'n/a')
        results.append(('✅', display_name, str(version)))
    except Exception as e:
        results.append(('❌', display_name, str(e)[:50]))

try_import('torch',                 'PyTorch')
try_import('transformers',          'Transformers')
try_import('sentence_transformers', 'Sentence-Transformers')
try_import('spacy',                 'spaCy')
try_import('stanza',                'Stanza')
try_import('bertopic',              'BERTopic')
try_import('gensim',                'Gensim')
try_import('easynmt',               'EasyNMT')
try_import('bert_score',            'BERTScore')
try_import('sacrebleu',             'SacreBLEU')
try_import('transformers', 'COMET (via HuggingFace)')
try_import('statsmodels',           'statsmodels')
try_import('pingouin',              'Pingouin')
try_import('krippendorff',          'Krippendorff')
try_import('ftfy',                  'ftfy')
try_import('langdetect',            'langdetect')
try_import('textstat',              'textstat')
try_import('sklearn',               'scikit-learn')
try_import('pandas',                'Pandas')
try_import('numpy',                 'NumPy')

table = Table(title='Library Import Status')
table.add_column('Status', style='bold')
table.add_column('Library')
table.add_column('Version')
for row in results:
    table.add_row(*row)
console.print(table)

failures = [r for r in results if r[0] == '❌']
if failures:
    print(f'\n⚠️  {len(failures)} import(s) failed')
else:
    print('\n✅ All imports successful')


## 6. End-to-End Smoke Test
Runs a German sustainable building sentence through the full pipeline.

In [ ]:
print('=== SMOKE TEST ===')

TEST_DE = (
    'Das Passivhaus-Konzept revolutioniert den Wohnungsbau in Deutschland. '
    'Durch hocheffiziente Wärmedämmung können Gebäude bis zu 90 Prozent '
    'Heizenergie einsparen. Die DGNB-Zertifizierung wurde 2018 eingeführt.'
)

# 1. Encoding fix
import ftfy
clean_text = ftfy.fix_text(TEST_DE)
print('[1/6] ftfy encoding fix ✅')

# 2. Language detection
from langdetect import detect
lang = detect(clean_text)
assert lang == 'de', f'Expected German, got {lang}'
print(f'[2/6] Language detected: {lang} ✅')

# 3. spaCy NER
import spacy
nlp = spacy.load('de_core_news_lg')
doc = nlp(clean_text)
entities = [(ent.text, ent.label_) for ent in doc.ents]
print(f'[3/6] spaCy NER: {entities} ✅')

# 4. Sentence embeddings
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embedding = embed_model.encode(clean_text)
print(f'[4/6] Embedding shape: {embedding.shape} ✅')

# 5. Translation
print('[5/6] Loading OPUS-MT (first run downloads ~300MB)...')
from easynmt import EasyNMT
translator = EasyNMT('opus-mt')
translation = translator.translate(clean_text, source_lang='de', target_lang='en')
print(f'[5/6] Translation: {translation[:80]}... ✅')

# 6. Readability
import textstat
fk = textstat.flesch_reading_ease(translation)
print(f'[6/6] Flesch score (EN): {fk:.1f} ✅')

print()
print('=== ✅ ALL SMOKE TESTS PASSED ===')


=== SMOKE TEST ===
[1/6] ftfy encoding fix ✅
[2/6] Language detected: de ✅
[3/6] spaCy NER: [('Passivhaus-Konzept', 'ORG'), ('Deutschland', 'LOC'), ('DGNB-Zertifizierung', 'ORG')] ✅
[4/6] Embedding shape: (384,) ✅
[5/6] Loading OPUS-MT (first run downloads ~300MB)...
[5/6] Translation: The passive house concept revolutionises housing construction in Germany. Thanks... ✅
[6/6] Flesch score (EN): 34.7 ✅

=== ✅ ALL SMOKE TESTS PASSED ===


## 7. Stanza German NER Test

In [ ]:
import stanza
print('Downloading Stanza German models (first run only, ~500MB)...')
stanza.download('de', processors='tokenize,ner', verbose=False)
stanza_nlp = stanza.Pipeline('de', processors='tokenize,ner', verbose=False)
stanza_doc = stanza_nlp(TEST_DE)
stanza_ents = [(ent.text, ent.type) for sent in stanza_doc.sentences for ent in sent.ents]
print(f'Stanza NER: {stanza_ents}')
print('✅ Stanza working')


Stanza NER: [('Deutschland', 'LOC'), ('DGNB', 'ORG')]
✅ Stanza working


## 8. HuggingFace Inference API Test

In [ ]:
from google.colab import userdata
from huggingface_hub import InferenceClient, login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)

primary_model  = config['llm']['primary_model']
fallback_model = config['llm']['fallback_model']
provider       = config['llm']['provider']

prompt = (
    'Classify this German text as: [energy_efficiency / materials / certification / policy / other].\n'
    f'Text: "{TEST_DE[:150]}"\n'
    'Answer with one label only:'
)

for model in [primary_model, fallback_model]:
    try:
        client = InferenceClient(provider=provider, token=hf_token)
        response = client.chat_completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20,
            temperature=0.0,
        )
        result = response.choices[0].message.content
        print(f'Model: {model}')
        print(f'Response: "{result.strip()}"')
        print('✅ HF Inference API working')
        break  # stop if successful
    except Exception as e:
        print(f'⚠️  {model} failed: {e}')
        print('   Trying fallback...')

TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

## 9. Save Reproducibility Snapshot

In [ ]:
import json, datetime, platform, sys, os
import torch, transformers, spacy

snapshot = {
    'timestamp':            datetime.datetime.now().isoformat(),
    'python_version':       sys.version,
    'platform':             platform.platform(),
    'torch_version':        torch.__version__,
    'transformers_version': transformers.__version__,
    'spacy_version':        spacy.__version__,
    'numpy_version':        np.__version__,
    'random_seed':          SEED,
    'project_root':         PROJECT_ROOT,
}

snapshot_path = f'{PROJECT_ROOT}/docs/environment_snapshot.json'
os.makedirs(os.path.dirname(snapshot_path), exist_ok=True)
with open(snapshot_path, 'w') as f:
    json.dump(snapshot, f, indent=2)

for k, v in snapshot.items():
    print(f'  {k:25s}: {v}')
print(f'\n✅ Snapshot saved to docs/environment_snapshot.json')
print()
print('🎉 Setup complete. Proceed to 01_corpus_audit.ipynb')
